# PBTA_RNA Clinical Data Analysis

**Exploratory analysis of pediatric brain tumor clinical data.**

Data: PBTA_RNA study


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/home/alon/menow_home_ass/PBTA_RNA"
PATIENT_FILE = f"{DATA_DIR}/data_clinical_patient_attributes.txt"
SAMPLE_FILE = f"{DATA_DIR}/data_clinical_sample_attributes.txt"

def read_patients():
    return pd.read_csv(PATIENT_FILE, sep="\t", header=4,
                       dtype={"AGE": float, "AGE_IN_DAYS": float,
                              "OS_MONTHS": float, "EFS_MONTHS": float})

def read_samples():
    return pd.read_csv(SAMPLE_FILE, sep="\t", header=4)

# OS/EFS cleaners
def clean_os(df):
    df = df.copy()
    df["OS_STATUS"] = df["OS_STATUS"].str.strip()
    df["os_label"] = df["OS_STATUS"].str.replace(r"^\d+:", "", regex=True)
    df["os_event"] = df["OS_STATUS"].apply(
        lambda x: 1 if pd.notna(x) and x.startswith("1:") else (0 if pd.notna(x) and x.startswith("0:") else np.nan))
    return df

def clean_efs(df):
    df = df.copy()
    df["EFS_STATUS"] = df["EFS_STATUS"].str.strip()
    df["efs_detail"] = df["EFS_STATUS"].str.replace(r"^\d+:", "", regex=True)
    df["efs_event"] = df["EFS_STATUS"].apply(
        lambda x: 0 if pd.notna(x) and x == "0:No Event"
        else (1 if pd.notna(x) and x != "1:NA" else np.nan))
    return df

# Categorical cleaners
def clean_race_eth(df):
    df = df.copy()
    df["RACE"] = df["RACE"].fillna("Unknown").replace({"Not Reported":"Unknown","Reported Unknown":"Unknown"})
    df["ETHNICITY"] = df["ETHNICITY"].fillna("Unknown").replace({"Not Reported":"Unknown","Reported Unknown":"Unknown"})
    return df

def clean_pred(df):
    df = df.copy()
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].fillna("Unknown")
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].replace("Not Reported","Unknown")
    df["CANCER_PREDISPOSITIONS"] = df["CANCER_PREDISPOSITIONS"].replace("None documented","No predisposition")
    return df

def clean_subtype(df):
    df = df.copy()
    df["MOLECULAR_SUBTYPE"] = df["MOLECULAR_SUBTYPE"].fillna("Unclassified")
    df["MOLECULAR_SUBTYPE"] = df["MOLECULAR_SUBTYPE"].replace("To be classified","Unclassified")
    return df

def clean_tf_tp(df):
    df = df.copy()
    df["TF_group"] = np.where(df["TUMOR_FRACTION"].isna(), "Unknown", "Measured")
    df["TP_group"] = np.where(df["TUMOR_PLOIDY"].isna(), "Unknown", "Measured")
    return df

# Kaplan-Meier helper
def kaplan_meier(times, events):
    d = pd.DataFrame({"t": times, "e": events}).dropna().sort_values("t")
    surv = 1.0
    n = len(d)
    res = []
    for t, grp in d.groupby("t", sort=False):
        ne = int(grp["e"].sum())
        if ne > 0:
            surv *= (1 - ne / n)
        res.append({"t": t, "s": surv, "n": n, "ne": ne})
        n -= len(grp)
    return pd.DataFrame(res)

def add_km(fig, km, label, color):
    fig.add_trace(go.Scatter(
        x=km["t"], y=km["s"], mode="lines", name=label,
        line=dict(color=color, width=2, shape="hv"),
        legendgroup=label,
        hovertemplate=f"Time: %{{x}}<br>Survival: %{{y:.3f}}<extra>{label}</extra>"))
    return fig

# Log-rank tests
def logrank2(t1,e1,t2,e2):
    from scipy.stats import chi2
    all_t = sorted(set(pd.concat([pd.Series(t1.dropna()),pd.Series(t2.dropna())]).dropna()))
    if len(all_t) < 2: return 1.0
    o1e=0; v=0
    d1=pd.DataFrame({"t":t1,"e":e1}).dropna()
    d2=pd.DataFrame({"t":t2,"e":e2}).dropna()
    for t in all_t:
        r1=(d1["t"]>=t).sum(); r2=(d2["t"]>=t).sum(); nr=r1+r2
        if nr==0: continue
        o1=int(((d1["t"]==t)&(d1["e"]==1)).sum())
        o2=int(((d2["t"]==t)&(d2["e"]==1)).sum())
        ot=o1+o2
        if ot==0: continue
        e1=ot*r1/nr; o1e+=(o1-e1)
        if nr>1: v+=ot*(r1/nr)*(r2/nr)*(nr-ot)/(nr-1)
    if v<=0: return 1.0
    return 1-chi2.cdf(o1e**2/v,1)

def logrank_multi(groups):
    from scipy.stats import chi2
    import numpy as np
    ng=len(groups)
    if ng<2: return 1.0
    all_t=sorted(set(pd.concat([pd.Series(g[0].dropna()) for g in groups]).dropna()))
    if len(all_t)<2: return 1.0
    O=np.zeros(ng);E=np.zeros(ng);V=np.zeros((ng,ng))
    for t in all_t:
        ar=np.array([(g[0]>=t).sum() for g in groups]); nr=ar.sum()
        if nr==0: continue
        ev=np.array([((g[0]==t)&(g[1]==1)).sum() for g in groups]); ot=ev.sum()
        if ot==0: continue
        O+=ev;E+=ot*ar/nr
        if nr>1:
            for i in range(ng):
                for j in range(ng):
                    if i==j: V[i,j]+=ot*ar[i]/nr*(1-ar[i]/nr)*(nr-ot)/(nr-1)
                    else: V[i,j]-=ot*ar[i]/nr*ar[j]/nr*(nr-ot)/(nr-1)
    try: return 1-chi2.cdf((O-E)@np.linalg.pinv(V)@(O-E),ng-1)
    except: return 1.0

print("Imports and helpers loaded.")


Imports and helpers loaded.


## Step 1: Load & Profile Patient Data

**Purpose:** Comprehensive overview of the patient dataset -- size, column types, missingness levels.


In [3]:
# Step 1: Load & Profile Patient Data -- Comprehensive overview of the patient dataset -- size, column types, missingness levels.
df = read_patients()
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns\n')
rows = []
for col in df.columns:
    nn = df[col].notna().sum(); nu = df[col].isna().sum(); pct = nu/len(df)*100
    if df[col].dtype == 'object': ex = f'unique={df[col].nunique()}'
    else: ex = f'min={df[col].min()}, max={df[col].max()}'
    rows.append({'Column':col,'Dtype':str(df[col].dtype),'Non-null':nn,'Null':nu,'% Null':f'{pct:.1f}%','Extra':ex})
print(pd.DataFrame(rows).to_string(index=False))


Shape: 2870 rows x 13 columns

                Column   Dtype  Non-null  Null % Null                                                                   Extra
            PATIENT_ID     str      2870     0   0.0%                                        min=PT_00G007DM, max=PT_ZZWYKZZ8
                   AGE float64      2812    58   2.0%                                                       min=0.0, max=73.0
           AGE_IN_DAYS float64      2812    58   2.0%                                                    min=1.0, max=26983.0
CANCER_PREDISPOSITIONS     str      2870     0   0.0% min=BRCA1/2,Other inherited conditions NOS, max=Von Hippel-Lindau (VHL)
            EFS_MONTHS float64      2054   816  28.4%                                                      min=0.0, max=246.0
            EFS_STATUS     str      2870     0   0.0%                    min=0:No Event, max=1:Second Malignancy - Metastatic
             ETHNICITY     str      2450   420  14.6%                            min=Hi

In [4]:
# Step 1: Load & Profile Patient Data -- Comprehensive overview of the patient dataset -- size, column types, missingness levels.
miss = df.isna().mean().sort_values(ascending=False)*100
fig = px.bar(x=miss.index,y=miss.values,title='Patient Data -- Missingness',labels={'x':'Column','y':'% Missing'},text=[f'{v:.1f}%' for v in miss.values])
fig.update_traces(marker_color='crimson',textposition='outside')
fig.update_layout(xaxis_tickangle=-45,height=450)
fig.show()


In [5]:
# Step 1: Load & Profile Patient Data -- Comprehensive overview of the patient dataset -- size, column types, missingness levels.
v = df['AGE'].isna().sum()
print(f'AGE has {v} missing ({v/len(df):.1%}) -- excluded from age plots')
print(f'OS_STATUS null: {df["OS_STATUS"].isna().sum()}')
print(f'EFS_STATUS null: {df["EFS_STATUS"].isna().sum()}')


AGE has 58 missing (2.0%) -- excluded from age plots
OS_STATUS null: 355
EFS_STATUS null: 0


## Step 2: Patient Demographics

**Purpose:** Age distribution, sex balance, race/ethnicity makeup.


In [6]:
# Step 2: Patient Demographics -- Age distribution, sex balance, race/ethnicity makeup.
df = read_patients()
df = clean_race_eth(df)
age = df.dropna(subset=['AGE'])
n_sex = age['SEX'].nunique(); n_race = age['RACE'].nunique(); n_eth = age['ETHNICITY'].nunique()
fig = go.Figure()
fig.add_trace(go.Histogram(x=age['AGE'],nbinsx=40,name='All',marker_color='steelblue',opacity=0.75))
colors = px.colors.qualitative.Plotly
for i,cat in enumerate(age['SEX'].value_counts().index):
    d = age[age['SEX']==cat]
    fig.add_trace(go.Histogram(x=d['AGE'],nbinsx=40,name=f'Sex:{cat}',marker_color=colors[i],opacity=0.6,visible=False))
for i,cat in enumerate(age['RACE'].value_counts().index):
    d = age[age['RACE']==cat]
    fig.add_trace(go.Histogram(x=d['AGE'],nbinsx=40,name=f'Race:{cat}',marker_color=colors[i%len(colors)],opacity=0.6,visible=False))
for i,cat in enumerate(age['ETHNICITY'].value_counts().index):
    d = age[age['ETHNICITY']==cat]
    fig.add_trace(go.Histogram(x=d['AGE'],nbinsx=40,name=f'Eth:{cat}',marker_color=colors[i%len(colors)],opacity=0.6,visible=False))
tr = 1 + n_sex + n_race + n_eth
fig.update_layout(updatemenus=[dict(buttons=[
    dict(label='Overall',method='update',args=[{'visible':[True]+[False]*(tr-1)},{'title':'Age -- Overall'}]),
    dict(label='By SEX',method='update',args=[{'visible':[True]+[True]*n_sex+[False]*(n_race+n_eth)},{'title':'Age by SEX'}]),
    dict(label='By RACE',method='update',args=[{'visible':[True]+[False]*n_sex+[True]*n_race+[False]*n_eth},{'title':'Age by RACE'}]),
    dict(label='By ETHNICITY',method='update',args=[{'visible':[True]+[False]*n_sex+[False]*n_race+[True]*n_eth},{'title':'Age by ETHNICITY'}])
],direction='down',showactive=True,x=1.0,y=1.15)],title='Age Distribution',xaxis_title='Age (years)',yaxis_title='Count',height=500,bargap=0.05)
fig.show()


In [7]:
# Step 2: Patient Demographics -- Age distribution, sex balance, race/ethnicity makeup.
fig2 = make_subplots(rows=2,cols=2,subplot_titles=('Sex','Race','Ethnicity','Age Stats'),specs=[[{'type':'bar'},{'type':'bar'}],[{'type':'bar'},{'type':'table'}]])
sc = df['SEX'].fillna('NaN').value_counts()
fig2.add_trace(go.Bar(x=sc.index,y=sc.values,marker_color='lightblue',text=sc.values,textposition='outside',showlegend=False),row=1,col=1)
rc = df['RACE'].value_counts()
fig2.add_trace(go.Bar(x=rc.index,y=rc.values,marker_color='lightgreen',text=rc.values,textposition='outside',showlegend=False),row=1,col=2)
ec = df['ETHNICITY'].value_counts()
fig2.add_trace(go.Bar(x=ec.index,y=ec.values,marker_color='lightskyblue',text=ec.values,textposition='outside',showlegend=False),row=2,col=1)
ac = df['AGE'].dropna()
tbl = pd.DataFrame([['Count',str(len(ac))],['Mean',f'{ac.mean():.1f}'],['Median',f'{ac.median():.1f}'],['Min',f'{ac.min():.1f}'],['Max',f'{ac.max():.1f}'],['Missing',str(df['AGE'].isna().sum())]],columns=['Stat','Value'])
fig2.add_trace(go.Table(header=dict(values=['Stat','Value'],fill_color='lightblue',align='left'),cells=dict(values=[tbl['Stat'],tbl['Value']],align='left',height=25)),row=2,col=2)
fig2.update_layout(height=600,title='Patient Demographics')
fig2.show()


In [8]:
# Step 2: Patient Demographics -- Age distribution, sex balance, race/ethnicity makeup.
print(f'RACE after: {sorted(df["RACE"].unique())}')
print(f'ETHNICITY after: {sorted(df["ETHNICITY"].unique())}')
print(f'Age data: {df["AGE"].notna().sum()}/{len(df)}')


RACE after: ['American Indian or Alaska Native', 'Asian', 'Black or African American', 'More Than One Race', 'Native Hawaiian or Other Pacific Islander', 'Other', 'Unknown', 'White']
ETHNICITY after: ['Hispanic or Latino', 'Not Hispanic or Latino', 'Unknown']
Age data: 2812/2870


## Step 3: Patient Survival Overview

**Purpose:** Outcome overview and Kaplan-Meier survival curves.


In [9]:
# Step 3: Patient Survival Overview -- Outcome overview and Kaplan-Meier survival curves.
df = read_patients()
df = clean_os(df)
df = clean_efs(df)
os_c = df['os_label'].value_counts(dropna=False).reset_index()
os_c.columns = ['Status','Count']
os_c['Status'] = os_c['Status'].fillna('Unknown')
fig = px.pie(os_c,values='Count',names='Status',title='Overall Survival Status',hole=0.3)
fig.update_traces(textinfo='label+percent')
fig.show()


In [10]:
# Step 3: Patient Survival Overview -- Outcome overview and Kaplan-Meier survival curves.
df['efs_bin'] = df['efs_event'].map({1:'Event',0:'No Event'}).fillna('Unknown')
fig = make_subplots(rows=1,cols=2,subplot_titles=('EFS Binary','EFS Detailed'))
eb = df['efs_bin'].value_counts().reset_index(); eb.columns=['Status','Count']
fig.add_trace(go.Bar(x=eb['Status'],y=eb['Count'],marker_color=['lightcoral','lightgreen','lightgray'],text=eb['Count'],textposition='outside',showlegend=False),row=1,col=1)
ed = df['efs_detail'].value_counts(dropna=False).reset_index(); ed.columns=['Status','Count']
ed['Status'] = ed['Status'].fillna('Unknown')
fig.add_trace(go.Bar(x=ed['Status'],y=ed['Count'],marker_color='lightcoral',text=ed['Count'],textposition='outside',showlegend=False),row=1,col=2)
fig.update_layout(title='Event-Free Survival Status',height=450,xaxis2_tickangle=-45)
fig.show()


In [11]:
# Step 3: Patient Survival Overview -- Outcome overview and Kaplan-Meier survival curves.
fig = make_subplots(rows=2,cols=1,subplot_titles=('OS -- KM','EFS -- KM'),vertical_spacing=0.15)
os_c = df[['OS_MONTHS','os_event']].dropna()
fig = add_km(fig, kaplan_meier(os_c['OS_MONTHS'],os_c['os_event']), 'OS', 'darkblue')
efs_c = df[['EFS_MONTHS','efs_event']].dropna()
km = kaplan_meier(efs_c['EFS_MONTHS'],efs_c['efs_event'])
fig.add_trace(go.Scatter(x=km['t'],y=km['s'],mode='lines',name='EFS',line=dict(color='darkred',width=2,shape='hv')),row=2,col=1)
fig.update_layout(height=600,title='KM Survival Curves')
fig.update_yaxes(range=[-0.05,1.05])
fig.show()


In [12]:
# Step 3: Patient Survival Overview -- Outcome overview and Kaplan-Meier survival curves.
co = df[['OS_MONTHS','os_event']].dropna()
ce = df[['EFS_MONTHS','efs_event']].dropna()
print(f'OS complete: {len(co)}/{len(df)}')
print(f'EFS complete: {len(ce)}/{len(df)}')
print(f'OS status:\n{df["OS_STATUS"].value_counts(dropna=False)}')
print(f'\nEFS event:\n{df["efs_event"].value_counts(dropna=False)}')


OS complete: 2096/2870
EFS complete: 2054/2870
OS status:
OS_STATUS
0:LIVING      1875
1:DECEASED     640
NaN            355
Name: count, dtype: int64

EFS event:
efs_event
0.0    1286
1.0    1222
NaN     362
Name: count, dtype: int64


## Step 4: Cancer Predispositions -- Prevalence & Demographics

**Purpose:** How common each predisposition is, and demographic patterns across predispositions.


In [13]:
# Step 4: Cancer Predispositions -- Prevalence & Demographics -- How common each predisposition is, and demographic patterns across predispositions.
import re
df = read_patients()
df = clean_pred(df)
df = clean_race_eth(df)
def explode(df):
    rows = []
    for _,r in df.iterrows():
        p = r['CANCER_PREDISPOSITIONS']
        if p in ('Unknown','No predisposition'): rows.append({**r,'pred':p})
        elif '),' in str(p):
            parts = re.split(r'\),\s*',str(p))
            for i,pr in enumerate(parts):
                if i < len(parts)-1: pr = pr + ')'
                rows.append({**r,'pred':pr.strip()})
        else: rows.append({**r,'pred':p})
    return pd.DataFrame(rows)
dex = explode(df)
total_p = df['PATIENT_ID'].nunique()
print(f'Patients: {total_p}, Exploded rows: {len(dex)}')
multi = df[df['CANCER_PREDISPOSITIONS'].str.contains(r'\),',na=False,regex=True)]
print(f'Multi-syndrome patients: {len(multi)}')


Patients: 2870, Exploded rows: 2886
Multi-syndrome patients: 13


In [14]:
# Step 4: Cancer Predispositions -- Prevalence & Demographics -- How common each predisposition is, and demographic patterns across predispositions.
pc = dex[~dex['pred'].isin(['No predisposition','Unknown'])]
pc = pc['pred'].value_counts().head(15).reset_index()
pc.columns = ['Pred','Count']
pc['%'] = (pc['Count']/total_p*100).round(1)
fig = px.bar(pc,y='Pred',x='Count',orientation='h',title=f'Top 15 Predispositions (N={total_p})',
             text=[f'{c} ({p:.1f}%)' for c,p in zip(pc['Count'],pc['%'])],color='Count',color_continuous_scale='Blues')
fig.update_traces(textposition='outside')
fig.update_layout(height=500,yaxis={'categoryorder':'total ascending'})
fig.show()


In [15]:
# Step 4: Cancer Predispositions -- Prevalence & Demographics -- How common each predisposition is, and demographic patterns across predispositions.
top10 = pc.head(10)
dp = top10['Pred'].iloc[0]
wp = dex[dex['pred']==dp]
wp2 = df[df['PATIENT_ID'].isin(wp['PATIENT_ID'].unique())]
fig = make_subplots(rows=2,cols=2,subplot_titles=('Age: With vs Without','Sex','Prevalence %','Summary'),
    specs=[[{'type':'box'},{'type':'bar'}],[{'type':'bar'},{'type':'table'}]])
fig.add_trace(go.Box(y=wp['AGE'].dropna(),name='With',marker_color=px.colors.qualitative.Plotly[0]),row=1,col=1)
fig.add_trace(go.Box(y=df[~df['PATIENT_ID'].isin(wp['PATIENT_ID'])]['AGE'].dropna(),name='Without',marker_color='lightgray'),row=1,col=1)
sx = wp2['SEX'].fillna('NaN').value_counts()
fig.add_trace(go.Bar(x=sx.index,y=sx.values,marker_color='lightcoral',text=sx.values,textposition='outside',showlegend=False),row=1,col=2)
fig.add_trace(go.Bar(x=top10['Pred'],y=top10['%'],marker_color='steelblue',text=top10['%'],textposition='outside',showlegend=False),row=2,col=1)
td = top10.rename(columns={'%':'Pct'})
fig.add_trace(go.Table(header=dict(values=list(td.columns),fill_color='lightblue',align='left'),
    cells=dict(values=[td[c] for c in td.columns],align='left',height=22)),row=2,col=2)
fig.update_layout(height=650,title='Predisposition Explorer')
fig.show()


In [16]:
# Step 4: Cancer Predispositions -- Prevalence & Demographics -- How common each predisposition is, and demographic patterns across predispositions.
ps = dex[~dex['pred'].isin(['No predisposition','Unknown'])]
ps = ps.groupby('pred').agg(Count=('PATIENT_ID','nunique'),MedAge=('AGE','median'),
    PctF=('SEX',lambda x: (x=='Female').sum()/len(x)*100 if len(x)>0 else 0)).reset_index().sort_values('Count',ascending=False)
ps['%Patients'] = (ps['Count']/total_p*100).round(1)
ps.columns = ['Predisposition','Count','Median Age','% Female','% Patients']
print(ps.to_string(index=False))


                                                                       Predisposition  Count  Median Age   % Female  % Patients
                                                     Neurofibromatosis, Type 1 (NF-1)    101        12.0  42.574257         3.5
                                                       Other inherited conditions NOS     57         9.0  43.859649         2.0
                                                          Li-Fraumeni syndrome (TP53)     35         6.0  62.857143         1.2
                                                     Neurofibromatosis, Type 2 (NF-2)     22        12.5  45.454545         0.8
                                                      Tuberous Sclerosis (TSC1, TSC2)     14         6.5  64.285714         0.5
Constitutional Mismatch Repair Deficiency Syndrome (biallelic PMS2, MLH1, MSH2, MSH6)      7         7.0  57.142857         0.2
                                              Lynch Syndrome (PMS2, MLH1, MSH2, MSH6)      7         7.0

## Step 5: Load & Profile Sample Data

**Purpose:** Overview of the sample-level dataset.


In [17]:
# Step 5: Load & Profile Sample Data -- Overview of the sample-level dataset.
df = read_samples()
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns\n')
rows = []
for col in df.columns:
    nn = df[col].notna().sum(); nu = df[col].isna().sum(); pct = nu/len(df)*100
    if df[col].dtype=='object': ex = f'unique={df[col].nunique()}'
    elif df[col].dtype in ('float64','int64'): ex = f'min={df[col].min():.2f}, max={df[col].max():.2f}'
    else: ex = ''
    rows.append({'Column':col,'Dtype':str(df[col].dtype),'Non-null':nn,'Null':nu,'% Null':f'{pct:.1f}%','Extra':ex})
print(pd.DataFrame(rows).to_string(index=False))


Shape: 4312 rows x 24 columns

                       Column   Dtype  Non-null  Null % Null              Extra
                   PATIENT_ID     str      4312     0   0.0%                   
                    SAMPLE_ID     str      4312     0   0.0%                   
              BROAD_HISTOLOGY     str      4265    47   1.1%                   
                 CANCER_GROUP     str      4074   238   5.5%                   
                  CANCER_TYPE     str      4015   297   6.9%                   
         CANCER_TYPE_DETAILED     str      4187   125   2.9%                   
              CBTN_TUMOR_TYPE     str      4312     0   0.0%                   
                   CNS_REGION     str      4179   133   3.1%                   
          COLLECTION_EVENT_ID     str      4312     0   0.0%                   
          EXPERIMENT_STRATEGY     str      4312     0   0.0%                   
    EXTENT_OF_TUMOR_RESECTION     str      4002   310   7.2%                   
     MATC

In [18]:
# Step 5: Load & Profile Sample Data -- Overview of the sample-level dataset.
miss = df.isna().mean().sort_values(ascending=False)*100
fig = px.bar(x=miss.index,y=miss.values,title='Sample Data -- Missingness',labels={'x':'Column','y':'% Missing'},
    text=[f'{v:.1f}%' for v in miss.values],color=miss.values,color_continuous_scale='Reds')
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=-45,height=500)
fig.show()
print('\nMissing > 0:'); print(miss[miss>0].round(1).to_string())



Missing > 0:
RNA_LIBRARY_SELECTION            42.5
TUMOR_FRACTION                   33.3
TUMOR_PLOIDY                     31.9
SAMPLE_TYPE                      23.7
MATCHED_NORMAL_SPECIMEN_ID       22.0
MATCHED_NORMAL_SAMPLE_ID         22.0
MOLECULAR_SUBTYPE                20.5
ONCOTREE_CODE                    19.9
EXTENT_OF_TUMOR_RESECTION         7.2
CANCER_TYPE                       6.9
CANCER_GROUP                      5.5
CNS_REGION                        3.1
CANCER_TYPE_DETAILED              2.9
PATHOLOGY_FREE_TEXT_DIAGNOSIS     2.8
BROAD_HISTOLOGY                   1.1


## Step 6: Sample Cancer Type Distributions

**Purpose:** Histological and anatomical breakdown of all samples.


In [19]:
# Step 6: Sample Cancer Type Distributions -- Histological and anatomical breakdown of all samples.
df = read_samples()
fig = make_subplots(rows=2,cols=2,subplot_titles=('Broad Histology','Cancer Group','CNS Region','Tumor Type'),
    specs=[[{'type':'bar'},{'type':'bar'}],[{'type':'bar'},{'type':'bar'}]])
hc = df['BROAD_HISTOLOGY'].value_counts().head(12)
fig.add_trace(go.Bar(x=hc.index,y=hc.values,marker_color='lightblue',text=hc.values,textposition='outside',showlegend=False),row=1,col=1)
cg = df['CANCER_GROUP'].value_counts().head(12)
fig.add_trace(go.Bar(x=cg.index,y=cg.values,marker_color='lightgreen',text=cg.values,textposition='outside',showlegend=False),row=1,col=2)
cr = df['CNS_REGION'].value_counts()
fig.add_trace(go.Bar(x=cr.index,y=cr.values,marker_color='lightsalmon',text=cr.values,textposition='outside',showlegend=False),row=2,col=1)
tt = df['TUMOR_TYPE'].value_counts()
comm = tt[tt>=30]; rare = tt[tt<30]
if len(rare)>0:
    oth = pd.Series({'Other':rare.sum()})
    tt_plot = pd.concat([comm,oth])
else: tt_plot = comm
fig.add_trace(go.Bar(x=tt_plot.index,y=tt_plot.values,marker_color='plum',text=tt_plot.values,textposition='outside',showlegend=False),row=2,col=2)
fig.update_layout(title='Sample Cancer Type Distributions',height=650,xaxis_tickangle=-45,xaxis2_tickangle=-45,xaxis3_tickangle=-45,xaxis4_tickangle=-45)
fig.show()


In [20]:
# Step 6: Sample Cancer Type Distributions -- Histological and anatomical breakdown of all samples.
print(f'Most common CG: {df["CANCER_GROUP"].value_counts().index[0]} ({df["CANCER_GROUP"].value_counts().iloc[0]})')
print(f'Total CGs: {df["CANCER_GROUP"].nunique()}')


Most common CG: Low-grade glioma (862)
Total CGs: 55


## Step 7: Tumor Purity & Ploidy

**Purpose:** Distribution of tumor purity and ploidy across samples.


In [21]:
# Step 7: Tumor Purity & Ploidy -- Distribution of tumor purity and ploidy across samples.
df = read_samples()
df = clean_tf_tp(df)
fig = make_subplots(rows=1,cols=3,subplot_titles=('Tumor Fraction','Tumor Ploidy','Fraction vs Ploidy'),
    specs=[[{'type':'histogram'},{'type':'histogram'},{'type':'scatter'}]])
tf = df[df['TF_group']=='Measured']['TUMOR_FRACTION'].dropna()
fig.add_trace(go.Histogram(x=tf,nbinsx=40,marker_color='steelblue',opacity=0.75),row=1,col=1)
nuk = (df['TF_group']=='Unknown').sum()
fig.add_annotation(text=f'Unknown: {nuk} ({nuk/len(df):.1%})',xref='paper',yref='paper',x=0.5,y=-0.3,showarrow=False,font=dict(color='gray'),row=1,col=1)
tp = df[df['TP_group']=='Measured']['TUMOR_PLOIDY'].dropna()
fig.add_trace(go.Histogram(x=tp,nbinsx=30,marker_color='darkgreen',opacity=0.75),row=1,col=2)
nuk2 = (df['TP_group']=='Unknown').sum()
fig.add_annotation(text=f'Unknown: {nuk2} ({nuk2/len(df):.1%})',xref='paper',yref='paper',x=0.5,y=-0.3,showarrow=False,font=dict(color='gray'),row=1,col=2)
sc = df.dropna(subset=['TUMOR_FRACTION','TUMOR_PLOIDY','CANCER_GROUP'])
top = sc['CANCER_GROUP'].value_counts().head(8).index
for cg in top:
    s = sc[sc['CANCER_GROUP']==cg]
    fig.add_trace(go.Scatter(x=s['TUMOR_FRACTION'],y=s['TUMOR_PLOIDY'],mode='markers',name=cg,marker=dict(size=5,opacity=0.6)),row=1,col=3)
fig.update_layout(height=450,title='Tumor Purity & Ploidy',xaxis3_title='Fraction',yaxis3_title='Ploidy')
fig.show()


In [22]:
# Step 7: Tumor Purity & Ploidy -- Distribution of tumor purity and ploidy across samples.
print(f'TF missing: {df["TUMOR_FRACTION"].isna().sum()}/{len(df)}')
print(f'TP missing: {df["TUMOR_PLOIDY"].isna().sum()}/{len(df)}')
print(f'TF range: {tf.min():.3f} - {tf.max():.3f}')
print(f'TP range: {tp.min():.1f} - {tp.max():.1f}')


TF missing: 1435/4312
TP missing: 1374/4312
TF range: 0.000 - 1.000
TP range: 2.0 - 4.0


## Step 8: Molecular Subtype Landscape

**Purpose:** The diversity of molecular subtypes and their relationship to cancer groups.


In [23]:
# Step 8: Molecular Subtype Landscape -- The diversity of molecular subtypes and their relationship to cancer groups.
df = read_samples()
df = clean_subtype(df)
sc = df['MOLECULAR_SUBTYPE'].value_counts().head(20).reset_index()
sc.columns = ['Subtype','Count']
sc['%'] = (sc['Count']/len(df)*100).round(1)
fig = px.bar(sc,y='Subtype',x='Count',orientation='h',title='Top 20 Subtypes',
    text=[f'{c} ({p:.1f}%)' for c,p in zip(sc['Count'],sc['%'])],color='Count',color_continuous_scale='Viridis')
fig.update_traces(textposition='outside')
fig.update_layout(height=600,yaxis={'categoryorder':'total ascending'})
fig.show()


In [24]:
# Step 8: Molecular Subtype Landscape -- The diversity of molecular subtypes and their relationship to cancer groups.
t15 = df['MOLECULAR_SUBTYPE'].value_counts().head(15).index
t10 = df['CANCER_GROUP'].value_counts().head(10).index
ct = pd.crosstab(df['MOLECULAR_SUBTYPE'],df['CANCER_GROUP'])
cf = ct.loc[ct.index.intersection(t15),ct.columns.intersection(t10)]
cn = cf.div(cf.sum(axis=1),axis=0).fillna(0)
fig = go.Figure(data=go.Heatmap(z=cn.values,x=cn.columns,y=cn.index,text=cf.values,texttemplate='%{text}',
    textfont=dict(size=9),colorscale='Blues',colorbar=dict(title='Proportion')))
fig.update_layout(title='Subtype x Cancer Group (Row %)',xaxis_tickangle=-45,height=550,yaxis=dict(autorange='reversed'))
fig.show()


In [25]:
# Step 8: Molecular Subtype Landscape -- The diversity of molecular subtypes and their relationship to cancer groups.
unc = (df['MOLECULAR_SUBTYPE']=='Unclassified').sum()
print(f'Unclassified: {unc} ({unc/len(df):.1%})')
print(f'Distinct subtypes: {df["MOLECULAR_SUBTYPE"].nunique()}')


Unclassified: 882 (20.5%)
Distinct subtypes: 133


## Step 9: Sequencing Strategy & RNA Library

**Purpose:** What sequencing methods and library prep were used.


In [26]:
# Step 9: Sequencing Strategy & RNA Library -- What sequencing methods and library prep were used.
df = read_samples()
fig = make_subplots(rows=1,cols=2,subplot_titles=('Experiment Strategy','RNA Library'))
es = df['EXPERIMENT_STRATEGY'].value_counts()
fig.add_trace(go.Bar(x=es.index,y=es.values,marker_color='teal',text=es.values,textposition='outside',showlegend=False),row=1,col=1)
lb = df['RNA_LIBRARY_SELECTION'].value_counts()
fig.add_trace(go.Bar(x=lb.index,y=lb.values,marker_color='purple',text=lb.values,textposition='outside',showlegend=False),row=1,col=2)
fig.update_layout(title='Sequencing Methods',height=400,xaxis_tickangle=-45,xaxis2_tickangle=-45)
fig.show()


In [27]:
# Step 9: Sequencing Strategy & RNA Library -- What sequencing methods and library prep were used.
print(f'Strategy unique: {df["EXPERIMENT_STRATEGY"].nunique()}')
print(f'Library unique: {df["RNA_LIBRARY_SELECTION"].nunique()}')
print('Strategies:',list(df['EXPERIMENT_STRATEGY'].unique()))


Strategy unique: 9
Library unique: 4
Strategies: ['WGS;RNA-Seq', 'Targeted Sequencing', 'WXS;RNA-Seq', 'Targeted Sequencing;Fusion_Panel', 'Fusion_Panel', 'WGS', 'Targeted Sequencing;RNA-Seq', 'RNA-Seq', 'WXS']


## Step 10: Merge Patient + Sample Data

**Purpose:** How well the two datasets connect.


In [28]:
# Step 10: Merge Patient + Sample Data -- How well the two datasets connect.
patients = read_patients()
samples = read_samples()
merged = samples.merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
print(f'Merged: {merged.shape}')
print(f'Samples: {len(samples)}, Patients: {len(patients)}')
sp = set(samples['PATIENT_ID'].unique())
pp = set(patients['PATIENT_ID'].unique())
print(f'Orphan samples: {len(sp-pp)}')
print(f'Orphan patients: {len(pp-sp)}')


Merged: (4312, 36)
Samples: 4312, Patients: 2870
Orphan samples: 0
Orphan patients: 0


In [29]:
# Step 10: Merge Patient + Sample Data -- How well the two datasets connect.
print(f'Samples without patient: {len(sp-pp)}')
print(f'Patients without sample: {len(pp-sp)}')
print('(AGE NaN is NOT a merge problem)')


Samples without patient: 0
Patients without sample: 0
(AGE NaN is NOT a merge problem)


## Step 11: Samples per Patient

**Purpose:** How many patients have single vs. multiple samples.


In [30]:
# Step 11: Samples per Patient -- How many patients have single vs. multiple samples.
samples = read_patients()
patients = read_patients()
merged = read_samples().merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
sc = merged.groupby('PATIENT_ID')['SAMPLE_ID'].nunique().reset_index()
sc.columns = ['PID','N']
fig = go.Figure()
fig.add_trace(go.Histogram(x=sc['N'],nbinsx=30,marker_color='darkorange',opacity=0.8))
fig.update_layout(title='Samples per Patient',xaxis_title='Samples',yaxis_title='Patients',height=400)
fig.show()
print('Top 10:'); print(sc.sort_values('N',ascending=False).head(10).to_string(index=False))


Top 10:
        PID  N
PT_Z4BF2NSB 15
PT_ZZRBX5JT 14
PT_HFQNKP5X 12
PT_EQX0VT4F 11
PT_KZ56XHJT 11
PT_GTHZF21E 10
PT_JXSTA5HB 10
PT_HJMP6PH2 10
PT_9BZETM0M 10
PT_9CQA1W10 10


In [31]:
# Step 11: Samples per Patient -- How many patients have single vs. multiple samples.
multi = (sc['N']>1).sum()
print(f'Patients with >1 sample: {multi} ({multi/len(sc):.1%})')
print(f'Mean: {sc["N"].mean():.2f}')


Patients with >1 sample: 840 (29.3%)
Mean: 1.50


## Step 12: Survival by Cancer Group

**Purpose:** OS and EFS stratified by cancer group.

**Scope:** All cancer groups are tested with a per-group log-rank (each group vs all other groups pooled; BH-FDR per endpoint). The plot draws a readable subset (groups with >=20 complete OS records, up to 15); legend labels show `(n=...)` = complete OS records behind that curve, with `*` (p<0.05) / `**` (FDR<0.05) marking a group whose OS differs from all other groups. The table below reports every group: sample/patient counts, median OS/EFS, and OS/EFS p & FDR q vs all others.


In [32]:
# Step 12: Survival by Cancer Group -- OS/EFS KM curves, per-group log-rank (each group vs all others).
patients = read_patients()
patients = clean_os(patients); patients = clean_efs(patients)
samples = read_samples()
merged = samples.merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
merged = merged[merged['CANCER_GROUP'].notna() & (merged['CANCER_GROUP'].astype(str).str.strip()!='')]

# BH-FDR (Benjamini-Hochberg) + significance labels
def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0: return np.array([])
    order = np.argsort(pvals)
    adj = pvals[order]*n/np.arange(1,n+1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(n); out[order] = np.clip(adj,0,1)
    return out

def star(p,q):
    """Table label: ** FDR<0.05, * p<0.05, ns otherwise."""
    if q<0.05: return '**'
    if p<0.05: return '*'
    return 'ns'

def legend_sig(p,q):
    """Legend suffix: only marks significant differences."""
    if q<0.05: return '**'
    if p<0.05: return '*'
    return ''

# Complete (time,event) pairs per group
osd = merged[['CANCER_GROUP','OS_MONTHS','os_event']].dropna()
efd = merged[['CANCER_GROUP','EFS_MONTHS','efs_event']].dropna()
os_n  = osd.groupby('CANCER_GROUP').size().sort_values(ascending=False)
efs_n = efd.groupby('CANCER_GROUP').size().sort_values(ascending=False)

MIN_N = 20      # minimum complete records to enter a per-group test
MAX_PLOT = 15   # maximum groups drawn
test_groups_os  = os_n[os_n>=MIN_N].index.tolist()
test_groups_efs = efs_n[efs_n>=MIN_N].index.tolist()
plot_groups = os_n[os_n>=MIN_N].head(MAX_PLOT).index.tolist()

# Per-group log-rank: group vs ALL other groups pooled (OS and EFS)
os_p = {g: logrank2(osd.loc[osd['CANCER_GROUP']==g,'OS_MONTHS'], osd.loc[osd['CANCER_GROUP']==g,'os_event'],
                    osd.loc[osd['CANCER_GROUP']!=g,'OS_MONTHS'], osd.loc[osd['CANCER_GROUP']!=g,'os_event'])
        for g in test_groups_os}
ef_p = {g: logrank2(efd.loc[efd['CANCER_GROUP']==g,'EFS_MONTHS'], efd.loc[efd['CANCER_GROUP']==g,'efs_event'],
                    efd.loc[efd['CANCER_GROUP']!=g,'EFS_MONTHS'], efd.loc[efd['CANCER_GROUP']!=g,'efs_event'])
        for g in test_groups_efs}
os_q = dict(zip(test_groups_os, bh_fdr([os_p[g] for g in test_groups_os])))
ef_q = dict(zip(test_groups_efs, bh_fdr([ef_p[g] for g in test_groups_efs])))

print(f'Cancer groups with complete OS: {len(os_n)} | EFS: {len(efs_n)}')
print(f'Tested vs others (N>={MIN_N}): {len(test_groups_os)} OS, {len(test_groups_efs)} EFS | plotted: {len(plot_groups)}')
print(f'FDR<0.05 vs all others: {sum(1 for g in test_groups_os if os_q[g]<0.05)}/{len(test_groups_os)} OS, '
      f'{sum(1 for g in test_groups_efs if ef_q[g]<0.05)}/{len(test_groups_efs)} EFS')

# KM plot: OS (top) + EFS (bottom); legend driven by OS, n = complete OS records
fig = make_subplots(rows=2,cols=1,subplot_titles=('OS by Cancer Group','EFS by Cancer Group'),vertical_spacing=0.15)
colors = (px.colors.qualitative.Set1 + px.colors.qualitative.Dark2 + px.colors.qualitative.Set3)
os_d=[]; ef_d=[]
for i,cg in enumerate(plot_groups):
    sub = merged[merged['CANCER_GROUP']==cg]
    os_s = sub[['OS_MONTHS','os_event']].dropna()
    label = f'{cg} (n={len(os_s)}){legend_sig(os_p.get(cg,1.0),os_q.get(cg,1.0))}'
    fig = add_km(fig,kaplan_meier(os_s['OS_MONTHS'],os_s['os_event']),label,colors[i%len(colors)])
    os_d.append((os_s['OS_MONTHS'],os_s['os_event']))
    ef_s = sub[['EFS_MONTHS','efs_event']].dropna()
    km = kaplan_meier(ef_s['EFS_MONTHS'],ef_s['efs_event'])
    fig.add_trace(go.Scatter(x=km['t'],y=km['s'],mode='lines',name=label,
        line=dict(color=colors[i%len(colors)],width=2,shape='hv'),legendgroup=label,showlegend=False),row=2,col=1)
    ef_d.append((ef_s['EFS_MONTHS'],ef_s['efs_event']))
if len(os_d)>=2:
    po=logrank_multi(os_d); pe=logrank_multi(ef_d)
    fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.02,text=f'global OS log-rank p={po:.4f}',showarrow=False,font=dict(size=11,color='darkblue'),row=1,col=1)
    fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.02,text=f'global EFS log-rank p={pe:.4f}',showarrow=False,font=dict(size=11,color='darkred'),row=2,col=1)
fig.update_layout(height=800,title='Survival by Cancer Group (n = complete OS records; * p<0.05, ** FDR<0.05 vs all other groups)',
                  legend=dict(font=dict(size=10)))
fig.update_yaxes(range=[-0.05,1.05])
fig.show()


Cancer groups with complete OS: 52 | EFS: 52
Tested vs others (N>=20): 22 OS, 22 EFS | plotted: 15
FDR<0.05 vs all others: 15/22 OS, 16/22 EFS


In [33]:
# Step 12: Per-group summary table for ALL cancer groups (sorted by N samples).
def fmt_p(p): return f'{p:.4g}' if p is not None else 'n/a'
rows = []
for cg in merged['CANCER_GROUP'].value_counts().index.tolist():
    s = merged[merged['CANCER_GROUP']==cg]
    os_s = s[['OS_MONTHS','os_event']].dropna()
    ef_s = s[['EFS_MONTHS','efs_event']].dropna()
    n_os = len(os_s); n_efs = len(ef_s)
    po = os_p.get(cg) if n_os>=MIN_N else None
    qo = os_q.get(cg) if n_os>=MIN_N else None
    pe = ef_p.get(cg) if n_efs>=MIN_N else None
    qe = ef_q.get(cg) if n_efs>=MIN_N else None
    rows.append({'Cancer Group':cg,'N samples':len(s),'N patients':s['PATIENT_ID'].nunique(),
        'N_OS':n_os,'Med OS':f'{os_s["OS_MONTHS"].median():.1f}' if n_os>0 else 'N/A',
        'OS p(vs others)':fmt_p(po),'OS q(FDR)':fmt_p(qo),'OS sig':star(po,qo) if (po is not None and qo is not None) else 'n/a',
        'N_EFS':n_efs,'Med EFS':f'{ef_s["EFS_MONTHS"].median():.1f}' if n_efs>0 else 'N/A',
        'EFS p(vs others)':fmt_p(pe),'EFS q(FDR)':fmt_p(qe),'EFS sig':star(pe,qe) if (pe is not None and qe is not None) else 'n/a'})
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
print('Significance: * p<0.05, ** FDR<0.05 (BH) -- log-rank of this group vs all other groups pooled.')


                                        Cancer Group  N samples  N patients  N_OS Med OS OS p(vs others) OS q(FDR) OS sig  N_EFS Med EFS EFS p(vs others) EFS q(FDR) EFS sig
                                    Low-grade glioma        862         622   712   56.0               0         0     **    710    29.0        1.477e-14  1.083e-13      **
                                   High-grade glioma        512         346   406   21.0               0         0     **    388    13.0        9.992e-16  1.099e-14      **
                                     Medulloblastoma        440         318   383   37.0          0.6121    0.6734     ns    383    32.0        3.127e-12  1.376e-11      **
                              Diffuse midline glioma        421         222   368   11.0               0         0     **    276     8.0                0          0      **
                                          Ependymoma        341         198   296   50.0          0.8223    0.8614     ns    296    22.

In [34]:
# Step 12: Validation -- multi-group patients and per-group test coverage.
mg = merged.groupby('PATIENT_ID')['CANCER_GROUP'].nunique()
m = mg[mg>1]
print(f'Patients in multiple curves: {len(m)} (entries: {int(m.sum())})')
print(f'Groups: {len(os_n)} with OS data, {len(efs_n)} with EFS data')
print(f'Per-group tests (N>={MIN_N}): {len(test_groups_os)} OS, {len(test_groups_efs)} EFS | plotted: {len(plot_groups)}')


Patients in multiple curves: 84 (entries: 172)
Groups: 52 with OS data, 52 with EFS data
Per-group tests (N>=20): 22 OS, 22 EFS | plotted: 15


## Step 13: Survival by Molecular Subtype (Global)

**Purpose:** OS and EFS stratified by molecular subtype.


In [35]:
# Step 13: Survival by Molecular Subtype (Global) -- OS and EFS stratified by molecular subtype.
patients = read_patients()
patients = clean_os(patients); patients = clean_efs(patients)
samples = read_samples(); samples = clean_subtype(samples)
merged = samples.merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
top6 = merged['MOLECULAR_SUBTYPE'].value_counts().head(6).index.tolist()
fig = make_subplots(rows=2,cols=1,subplot_titles=('OS by Subtype','EFS by Subtype'),vertical_spacing=0.15)
colors = px.colors.qualitative.Set1 + px.colors.qualitative.Set2
os_d=[]; ef_d=[]
for i,st in enumerate(top6):
    sub = merged[merged['MOLECULAR_SUBTYPE']==st]
    os_s = sub[['OS_MONTHS','os_event']].dropna()
    if len(os_s)>5:
        fig = add_km(fig,kaplan_meier(os_s['OS_MONTHS'],os_s['os_event']),st,colors[i%len(colors)])
        os_d.append((os_s['OS_MONTHS'],os_s['os_event']))
    ef_s = sub[['EFS_MONTHS','efs_event']].dropna()
    if len(ef_s)>5:
        km = kaplan_meier(ef_s['EFS_MONTHS'],ef_s['efs_event'])
        fig.add_trace(go.Scatter(x=km['t'],y=km['s'],mode='lines',name=st,
            line=dict(color=colors[i%len(colors)],width=2,shape='hv'),legendgroup=st,showlegend=False),row=2,col=1)
        ef_d.append((ef_s['EFS_MONTHS'],ef_s['efs_event']))
if len(os_d)>=2:
    fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.02,text=f'OS p={logrank_multi(os_d):.4f}',showarrow=False,font=dict(size=11,color='darkblue'),row=1,col=1)
    fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.02,text=f'EFS p={logrank_multi(ef_d):.4f}',showarrow=False,font=dict(size=11,color='darkred'),row=2,col=1)
fig.update_layout(height=650,title='Survival by Subtype (Global)')
fig.update_yaxes(range=[-0.05,1.05])
fig.show()


In [36]:
# Step 13: Survival by Molecular Subtype (Global) -- OS and EFS stratified by molecular subtype.
# Per-cancer-group subtype analysis
div = merged.groupby('CANCER_GROUP')['MOLECULAR_SUBTYPE'].nunique().sort_values(ascending=False)
print('Subtype diversity:\n',div)
rich = div[div>=5].index.tolist()
print(f'\nGroups with >=5 subtypes: {rich}')
for cg in rich[:3]:
    cgm = merged[merged['CANCER_GROUP']==cg]
    top = cgm['MOLECULAR_SUBTYPE'].value_counts().head(5).index.tolist()
    if len(top)<2: continue
    fig = make_subplots(rows=2,cols=1,subplot_titles=(f'OS -- {cg}',f'EFS -- {cg}'),vertical_spacing=0.15)
    os_d2=[]; ef_d2=[]
    for j,st in enumerate(top):
        sub = cgm[cgm['MOLECULAR_SUBTYPE']==st]
        os_s = sub[['OS_MONTHS','os_event']].dropna()
        if len(os_s)>3:
            fig = add_km(fig,kaplan_meier(os_s['OS_MONTHS'],os_s['os_event']),st,colors[j%len(colors)])
            os_d2.append((os_s['OS_MONTHS'],os_s['os_event']))
        ef_s = sub[['EFS_MONTHS','efs_event']].dropna()
        if len(ef_s)>3:
            km = kaplan_meier(ef_s['EFS_MONTHS'],ef_s['efs_event'])
            fig.add_trace(go.Scatter(x=km['t'],y=km['s'],mode='lines',name=st,
                line=dict(color=colors[j%len(colors)],width=2,shape='hv'),legendgroup=st,showlegend=False),row=2,col=1)
            ef_d2.append((ef_s['EFS_MONTHS'],ef_s['efs_event']))
    if len(os_d2)>=2:
        fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.02,text=f'OS p={logrank_multi(os_d2):.4f}',showarrow=False,font=dict(size=10,color='darkblue'),row=1,col=1)
        fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.02,text=f'EFS p={logrank_multi(ef_d2):.4f}',showarrow=False,font=dict(size=10,color='darkred'),row=2,col=1)
    fig.update_layout(height=550,title=f'Survival by Subtype -- {cg}')
    fig.update_yaxes(range=[-0.05,1.05])
    fig.show()


Subtype diversity:
 CANCER_GROUP
Low-grade glioma                                        43
Ganglioglioma                                           18
Glial-neuronal tumor NOS                                15
Ependymoma                                               8
Medulloblastoma                                          6
Infant-type hemispheric glioma                           6
High-grade glioma                                        5
Pineoblastoma                                            4
Atypical Teratoid Rhabdoid Tumor                         4
Ganglioneuroblastoma                                     3
Neuroblastoma                                            3
Desmoplastic infantile astrocytoma and ganglioglioma     3
Chordoma                                                 3
Pleomorphic xanthoastrocytoma                            3
Rosette-forming glioneuronal tumor                       2
Embryonal tumor with multilayer rosettes                 2
Diffuse leptomeningeal 

In [37]:
# Step 13: Survival by Molecular Subtype (Global) -- OS and EFS stratified by molecular subtype.
print('Top 10 subtypes:')
print(merged['MOLECULAR_SUBTYPE'].value_counts().head(10))
print('\nDiversity:',div)


Top 10 subtypes:
MOLECULAR_SUBTYPE
Unclassified              882
HGG, H3 wildtype          348
LGG, KIAA1549-BRAF        337
DMG, H3 K28, TP53         246
DMG, H3 K28               175
MB, Group4                151
LGG, wildtype             146
EPN, To be classified     140
CRANIO, ADAM              115
HGG, H3 wildtype, TP53    115
Name: count, dtype: int64

Diversity: CANCER_GROUP
Low-grade glioma                                        43
Ganglioglioma                                           18
Glial-neuronal tumor NOS                                15
Ependymoma                                               8
Medulloblastoma                                          6
Infant-type hemispheric glioma                           6
High-grade glioma                                        5
Pineoblastoma                                            4
Atypical Teratoid Rhabdoid Tumor                         4
Ganglioneuroblastoma                                     3
Neuroblastoma           

## Step 14: Age at Diagnosis by Cancer Group

**Purpose:** Whether different cancer groups occur at different ages.


In [38]:
# Step 14: Age at Diagnosis by Cancer Group -- Whether different cancer groups occur at different ages.
patients = read_patients()
samples = read_samples()
merged = samples.merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
top8 = merged['CANCER_GROUP'].value_counts().head(8).index.tolist()
pd8 = merged[merged['CANCER_GROUP'].isin(top8)].dropna(subset=['AGE'])
fig = go.Figure()
for i,cg in enumerate(top8):
    sub = pd8[pd8['CANCER_GROUP']==cg]['AGE']
    fig.add_trace(go.Box(y=sub,name=cg,boxmean='sd',marker_color=px.colors.qualitative.Plotly[i]))
groups = [merged[merged['CANCER_GROUP']==cg]['AGE'].dropna() for cg in top8]
stat,p_kw = kruskal(*groups)
fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.05,text=f'KW: H={stat:.2f}, p={p_kw:.4f}',showarrow=False,font=dict(size=12,color='darkred'))
fig.update_layout(title='Age at Diagnosis by Cancer Group',yaxis_title='Age (years)',height=500,xaxis_tickangle=-45)
fig.show()
if p_kw<0.05:
    print('Post-hoc (MW p<0.01):')
    for i,c1 in enumerate(top8):
        for j,c2 in enumerate(top8):
            if i>=j: continue
            g1 = merged[merged['CANCER_GROUP']==c1]['AGE'].dropna()
            g2 = merged[merged['CANCER_GROUP']==c2]['AGE'].dropna()
            if len(g1)>5 and len(g2)>5:
                _,p = mannwhitneyu(g1,g2,alternative='two-sided')
                if p<0.01: print(f'  {c1:35s} vs {c2:35s}: p={p:.6f}')


Post-hoc (MW p<0.01):
  Low-grade glioma                    vs High-grade glioma                  : p=0.000000
  Low-grade glioma                    vs Ependymoma                         : p=0.000000
  Low-grade glioma                    vs Atypical Teratoid Rhabdoid Tumor   : p=0.000000
  High-grade glioma                   vs Medulloblastoma                    : p=0.000000
  High-grade glioma                   vs Diffuse midline glioma             : p=0.000000
  High-grade glioma                   vs Ependymoma                         : p=0.000000
  High-grade glioma                   vs Atypical Teratoid Rhabdoid Tumor   : p=0.000000
  High-grade glioma                   vs Ganglioglioma                      : p=0.000813
  High-grade glioma                   vs Adamantinomatous Craniopharyngioma : p=0.000002
  Medulloblastoma                     vs Ependymoma                         : p=0.000008
  Medulloblastoma                     vs Atypical Teratoid Rhabdoid Tumor   : p=0.000000

In [39]:
# Step 14: Age at Diagnosis by Cancer Group -- Whether different cancer groups occur at different ages.
print(f'KW: H={stat:.2f}, p={p_kw:.4f}')
print('Group sizes:',[len(g) for g in groups])


KW: H=352.27, p=0.0000
Group sizes: [861, 475, 436, 410, 340, 152, 151, 115]


## Step 15: Sex Balance by Cancer Group

**Purpose:** Whether certain cancer groups show sex bias.


In [40]:
# Step 15: Sex Balance by Cancer Group -- Whether certain cancer groups show sex bias.
patients = read_patients()
samples = read_samples()
merged = samples.merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
top8 = merged['CANCER_GROUP'].value_counts().head(8).index.tolist()
pd8 = merged[merged['CANCER_GROUP'].isin(top8)].copy()
pd8['SEX'] = pd8['SEX'].fillna('Unknown')
ct = pd.crosstab(pd8['CANCER_GROUP'],pd8['SEX'])
fig = go.Figure()
for sex in ct.columns:
    fig.add_trace(go.Bar(name=sex,x=ct.index,y=ct[sex],text=ct[sex],textposition='inside'))
chi2,p,_,_ = chi2_contingency(ct)
fig.add_annotation(xref='paper',yref='paper',x=0.5,y=1.05,text=f'Chi2: x2={chi2:.2f}, p={p:.4f}',showarrow=False,font=dict(size=12,color='darkred'))
fig.update_layout(barmode='stack',title='Sex by Cancer Group',xaxis_tickangle=-45,height=450)
fig.show()


In [41]:
# Step 15: Sex Balance by Cancer Group -- Whether certain cancer groups show sex bias.
ct = pd.crosstab(merged['CANCER_GROUP'],merged['SEX'])
chi2,p,dof,_ = chi2_contingency(ct.fillna(0))
print(f'Chi2: x2={chi2:.2f}, p={p:.4f}, df={dof}')


Chi2: x2=157.14, p=0.0014, df=108


## Step 16: Purity by Cancer Group & Tumor Type

**Purpose:** Tumor purity differences across cancer groups and clinical states.


In [42]:
# Step 16: Purity by Cancer Group & Tumor Type -- Tumor purity differences across cancer groups and clinical states.
patients = read_patients()
samples = read_samples(); samples = clean_tf_tp(samples)
merged = samples.merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
fig = make_subplots(rows=1,cols=2,subplot_titles=('TF by Cancer Group','TF by Tumor Type'))
top8 = merged['CANCER_GROUP'].value_counts().head(8).index.tolist()
for i,cg in enumerate(top8):
    sub = merged[merged['CANCER_GROUP']==cg]['TUMOR_FRACTION'].dropna()
    if len(sub)>0: fig.add_trace(go.Box(y=sub,name=cg,boxmean='sd',marker_color=px.colors.qualitative.Plotly[i]),row=1,col=1)
top_tt = ['primary','metastatic','progression','recurrence']
for tt in top_tt:
    sub = merged[merged['TUMOR_TYPE']==tt]['TUMOR_FRACTION'].dropna()
    if len(sub)>0: fig.add_trace(go.Box(y=sub,name=tt,boxmean='sd',marker_color=px.colors.qualitative.Set2[top_tt.index(tt)]),row=1,col=2)
fig.update_layout(height=500,title='Tumor Fraction Analysis',yaxis_title='Fraction',xaxis_tickangle=-45,xaxis2_tickangle=-45)
fig.show()
for name,grps in [('Cancer Group',top8),('Tumor Type',top_tt)]:
    g = [merged[merged['CANCER_GROUP']==cg]['TUMOR_FRACTION'].dropna() for cg in grps if len(merged[merged['CANCER_GROUP']==cg]['TUMOR_FRACTION'].dropna())>5]
    if len(g)>=2:
        s,p = kruskal(*g); print(f'KW ({name}): H={s:.2f}, p={p:.4f}')


KW (Cancer Group): H=76.89, p=0.0000


## Step 17: Predisposition vs Outcome

**Purpose:** Whether known cancer predisposition affects survival or age of onset.


In [43]:
# Step 17: Predisposition vs Outcome -- Whether known cancer predisposition affects survival or age of onset.
patients = read_patients()
patients = clean_pred(patients); patients = clean_os(patients); patients = clean_efs(patients)
patients['hp'] = ~patients['CANCER_PREDISPOSITIONS'].isin(['No predisposition','Unknown'])
print(f'With predisposition: {patients["hp"].sum()} / {len(patients)}')
fig = make_subplots(rows=2,cols=2,subplot_titles=('OS by Pred','EFS by Pred','Age by Pred','OS Event'),
    specs=[[{'type':'scatter'},{'type':'scatter'}],[{'type':'box'},{'type':'bar'}]])
for i,(lab,has) in enumerate([('No',False),('Yes',True)]):
    sub = patients[patients['hp']==has]
    os_s = sub[['OS_MONTHS','os_event']].dropna()
    if len(os_s)>3:
        fig = add_km(fig,kaplan_meier(os_s['OS_MONTHS'],os_s['os_event']),lab,px.colors.qualitative.Set1[i])
    ef_s = sub[['EFS_MONTHS','efs_event']].dropna()
    if len(ef_s)>3:
        km = kaplan_meier(ef_s['EFS_MONTHS'],ef_s['efs_event'])
        fig.add_trace(go.Scatter(x=km['t'],y=km['s'],mode='lines',name=lab,
            line=dict(color=px.colors.qualitative.Set1[i],width=2,shape='hv'),legendgroup=lab,showlegend=False),row=1,col=2)
    fig.add_trace(go.Box(y=sub['AGE'].dropna(),name=lab,boxmean='sd',marker_color=px.colors.qualitative.Set1[i]),row=2,col=1)
fig.update_layout(height=600,title='Predisposition vs Outcome')
fig.show()


With predisposition: 245 / 2870


In [44]:
# Step 17: Predisposition vs Outcome -- Whether known cancer predisposition affects survival or age of onset.
t = patients[patients['hp']==True]; f = patients[patients['hp']==False]
os_t = t[['OS_MONTHS','os_event']].dropna(); os_f = f[['OS_MONTHS','os_event']].dropna()
if len(os_t)>3 and len(os_f)>3: print(f'Log-rank OS: p={logrank2(os_t["OS_MONTHS"],os_t["os_event"],os_f["OS_MONTHS"],os_f["os_event"]):.4f}')
ef_t = t[['EFS_MONTHS','efs_event']].dropna(); ef_f = f[['EFS_MONTHS','efs_event']].dropna()
if len(ef_t)>3 and len(ef_f)>3: print(f'Log-rank EFS: p={logrank2(ef_t["EFS_MONTHS"],ef_t["efs_event"],ef_f["EFS_MONTHS"],ef_f["efs_event"]):.4f}')
at = t['AGE'].dropna(); af = f['AGE'].dropna()
if len(at)>3 and len(af)>3:
    s,p = mannwhitneyu(at,af,alternative='two-sided')
    print(f'MW Age: U={s:.0f}, p={p:.4f}, medians: {at.median():.1f} vs {af.median():.1f}')


Log-rank OS: p=0.3995
Log-rank EFS: p=0.0649
MW Age: U=330282, p=0.1093, medians: 9.0 vs 8.0


## Step 18: CNS Region vs Cancer Group

**Purpose:** Anatomical distribution patterns of different cancer types.


In [45]:
# Step 18: CNS Region vs Cancer Group -- Anatomical distribution patterns of different cancer types.
patients = read_patients(); samples = read_samples()
merged = samples.merge(patients,on='PATIENT_ID',how='left',suffixes=('','_p'))
ct = pd.crosstab(merged['CNS_REGION'],merged['CANCER_GROUP'])
tr = merged['CNS_REGION'].value_counts().head(8).index
tc = merged['CANCER_GROUP'].value_counts().head(10).index
cf = ct.loc[ct.index.intersection(tr),ct.columns.intersection(tc)]
cn = cf.div(cf.sum(axis=1),axis=0).fillna(0)
fig = go.Figure(data=go.Heatmap(z=cn.values,x=cn.columns,y=cn.index,text=cf.values,texttemplate='%{text}',
    textfont=dict(size=10),colorscale='Purples',colorbar=dict(title='Proportion')))
fig.update_layout(title='CNS Region x Cancer Group (Row-Normalized)',xaxis_tickangle=-45,height=550)
fig.show()


In [46]:
# Step 18: CNS Region vs Cancer Group -- Anatomical distribution patterns of different cancer types.
print(f'CNS_REGION missing: {merged["CNS_REGION"].isna().sum()}')
print(merged['CNS_REGION'].value_counts().head(8))


CNS_REGION missing: 133
CNS_REGION
Hemispheric        1017
Posterior fossa     866
Mixed               763
Midline             508
Other               379
Ventricles          220
Spine               219
Suprasellar         174
Name: count, dtype: int64


## Step 19: Generate Summary Report

**Purpose:** A markdown summary of all findings.


In [47]:
# Step 19: Generate Summary Report -- A markdown summary of all findings.
patients = read_patients(); samples = read_samples()
patients = clean_os(patients); patients = clean_efs(patients); patients = clean_pred(patients)
samples = clean_subtype(samples)
n_p = patients['PATIENT_ID'].nunique(); n_s = samples['SAMPLE_ID'].nunique()
n_cg = samples['CANCER_GROUP'].nunique()
a_m = patients['AGE'].median(); a_r = (patients['AGE'].min(),patients['AGE'].max())
p_m = (patients['SEX']=='Male').mean()*100; p_f = (patients['SEX']=='Female').mean()*100
n_pr = (patients['CANCER_PREDISPOSITIONS']!='No predisposition').sum()
t_cg = samples['CANCER_GROUP'].value_counts().index[0]; t_cg_c = samples['CANCER_GROUP'].value_counts().iloc[0]
n_uc = (samples['MOLECULAR_SUBTYPE']=='Unclassified').sum(); n_mt = samples['TUMOR_FRACTION'].isna().sum(); n_mtp = samples['TUMOR_PLOIDY'].isna().sum()
os_c = patients[['OS_MONTHS','os_event']].dropna(); os_m = os_c['OS_MONTHS'].median()
report = f'''# PBTA_RNA Basic Clinical Summary

## Dataset Overview
- Patients: {n_p}
- Samples: {n_s}
- Cancer Groups: {n_cg}
- Most common: {t_cg} ({t_cg_c})

## Demographics
- Age: median {a_m:.1f}y, range {a_r[0]:.0f}-{a_r[1]:.0f}
- Sex: {p_m:.1f}% M, {p_f:.1f}% F

## Survival
- Median OS: {os_m:.1f}mo
- Patients with OS data: {len(os_c)}/{n_p}

## Predispositions
- Known predisposition: {n_pr}/{n_p} ({n_pr/n_p*100:.1f}%)

## Subtypes
- Unclassified: {n_uc}/{n_s} ({n_uc/n_s*100:.1f}%)
- Distinct subtypes: {samples['MOLECULAR_SUBTYPE'].nunique()}

## Tumor Purity
- Missing TF: {n_mt}/{n_s} ({n_mt/n_s*100:.1f}%)
- Missing TP: {n_mtp}/{n_s} ({n_mtp/n_s*100:.1f}%)
'''
with open('/home/alon/menow_home_ass/basic_clinical_summary.md','w') as f: f.write(report)
print('Saved: basic_clinical_summary.md')
print(report)


Saved: basic_clinical_summary.md
# PBTA_RNA Basic Clinical Summary

## Dataset Overview
- Patients: 2870
- Samples: 4312
- Cancer Groups: 55
- Most common: Low-grade glioma (862)

## Demographics
- Age: median 8.0y, range 0-73
- Sex: 53.7% M, 45.7% F

## Survival
- Median OS: 39.0mo
- Patients with OS data: 2096/2870

## Predispositions
- Known predisposition: 313/2870 (10.9%)

## Subtypes
- Unclassified: 882/4312 (20.5%)
- Distinct subtypes: 133

## Tumor Purity
- Missing TF: 1435/4312 (33.3%)
- Missing TP: 1374/4312 (31.9%)



## Step 20: Summary Table of All Figures

**Purpose:** Consolidated overview of every figure produced.


In [48]:
# Step 20: Summary Table of All Figures -- Consolidated overview of every figure produced.
figures = [
    (1,'Missingness Bar','Bar','Patient column missingness'),
    (2,'Age Histogram (dropdown)','Histogram','Age by SEX/RACE/ETHNICITY'),
    (2,'Demographics Grid','Bar+Table','Sex, race, ethnicity, age stats'),
    (3,'OS Pie','Pie','Deceased vs Living'),
    (3,'EFS Binary+Detailed Bars','Bar','EFS categories'),
    (3,'OS KM','KM','OS survival probability'),
    (3,'EFS KM','KM','EFS survival probability'),
    (4,'Predisposition Prevalence','Bar','Top 15 syndromes'),
    (4,'Predisposition Explorer','Multi','Age/sex/prevalence per syndrome'),
    (5,'Sample Missingness Bar','Bar','Missing data in samples'),
    (6,'Cancer Types Grid','Bar','Histology, group, region, type'),
    (7,'TF Histogram','Histogram','Purity distribution'),
    (7,'TP Histogram','Histogram','Ploidy distribution'),
    (7,'TF vs TP Scatter','Scatter','Purity vs ploidy by group'),
    (8,'Subtype Bar','Bar','Top 20 subtypes'),
    (8,'Subtype x Group Heatmap','Heatmap','Subtype-group cross-tab'),
    (9,'Strategy Bar','Bar','Experiment strategies'),
    (9,'Library Bar','Bar','Library prep methods'),
    ('9a','Groups per Patient','Bar','Distinct CGs per patient'),
    ('9a','Co-occurrence Heatmap','Heatmap','CG pair patterns'),
    (11,'Samples/Patient Histogram','Histogram','Samples per patient'),
    (12,'OS by CG KM','KM','OS by top 6 CGs'),
    (12,'EFS by CG KM','KM','EFS by top 6 CGs'),
    (13,'OS by Subtype KM','KM','OS by top 6 subtypes'),
    (13,'EFS by Subtype KM','KM','EFS by top 6 subtypes'),
    (13,'Subtype KM per Group','KM','Subtype survival in rich groups'),
    (14,'Age by Group Box','Box','Age distribution with KW'),
    (15,'Sex by Group Bar','Bar','Sex balance with chi2'),
    (16,'TF by Group Box','Box','Purity across groups'),
    (16,'TF by Type Box','Box','Purity by clinical state'),
    (17,'Predisposition KM','KM','OS/EFS by pred status'),
    (17,'Age by Pred Box','Box','Age by pred status'),
    (18,'CNS x CG Heatmap','Heatmap','Anatomical distribution'),
]
fd = pd.DataFrame(figures,columns=['Step','Title','Type','Insight'])
fd['Interactive'] = 'Yes'
print(f'Total figures: {len(fd)}\n')
print(fd.to_string(index=False))


Total figures: 33

Step                     Title      Type                         Insight Interactive
   1           Missingness Bar       Bar      Patient column missingness         Yes
   2  Age Histogram (dropdown) Histogram       Age by SEX/RACE/ETHNICITY         Yes
   2         Demographics Grid Bar+Table Sex, race, ethnicity, age stats         Yes
   3                    OS Pie       Pie              Deceased vs Living         Yes
   3  EFS Binary+Detailed Bars       Bar                  EFS categories         Yes
   3                     OS KM        KM         OS survival probability         Yes
   3                    EFS KM        KM        EFS survival probability         Yes
   4 Predisposition Prevalence       Bar                Top 15 syndromes         Yes
   4   Predisposition Explorer     Multi Age/sex/prevalence per syndrome         Yes
   5    Sample Missingness Bar       Bar         Missing data in samples         Yes
   6         Cancer Types Grid       Bar  Hist

In [49]:
# Step 20: Summary Table of All Figures -- Consolidated overview of every figure produced.
steps_summary = [
    (1,'Load & Profile Patient Data'),(2,'Patient Demographics'),(3,'Patient Survival Overview'),
    (4,'Cancer Predispositions'),(5,'Load & Profile Sample Data'),(6,'Sample Cancer Types'),
    (7,'Tumor Purity & Ploidy'),(8,'Molecular Subtype Landscape'),(9,'Sequencing Strategy'),
    ('9a','Multi-Cancer-Group Analysis'),(10,'Merge Patient + Sample Data'),(11,'Samples per Patient'),
    (12,'Survival by Cancer Group'),(13,'Survival by Molecular Subtype'),(14,'Age by Cancer Group'),
    (15,'Sex Balance by Cancer Group'),(16,'Purity by Group & Type'),
    (17,'Predisposition vs Outcome'),(18,'CNS Region vs Cancer Group'),
    (19,'Generate Summary Report'),(20,'Summary Table of All Figures'),
]
print('='*60)
print('NOTEBOOK COMPLETE -- All 20 steps implemented')
print('='*60)
for s,n in steps_summary: print(f'  Step {str(s):3s}: {n}')


NOTEBOOK COMPLETE -- All 20 steps implemented
  Step 1  : Load & Profile Patient Data
  Step 2  : Patient Demographics
  Step 3  : Patient Survival Overview
  Step 4  : Cancer Predispositions
  Step 5  : Load & Profile Sample Data
  Step 6  : Sample Cancer Types
  Step 7  : Tumor Purity & Ploidy
  Step 8  : Molecular Subtype Landscape
  Step 9  : Sequencing Strategy
  Step 9a : Multi-Cancer-Group Analysis
  Step 10 : Merge Patient + Sample Data
  Step 11 : Samples per Patient
  Step 12 : Survival by Cancer Group
  Step 13 : Survival by Molecular Subtype
  Step 14 : Age by Cancer Group
  Step 15 : Sex Balance by Cancer Group
  Step 16 : Purity by Group & Type
  Step 17 : Predisposition vs Outcome
  Step 18 : CNS Region vs Cancer Group
  Step 19 : Generate Summary Report
  Step 20 : Summary Table of All Figures


## Step 9a: Multi-Cancer-Group Analysis

**Purpose:** How many patients have samples in different cancer groups.


In [50]:
# Step 9a: Multi-Cancer-Group Analysis -- How many patients have samples in different cancer groups.
df = read_samples()
pg = df.groupby('PATIENT_ID')['CANCER_GROUP'].apply(set).reset_index()
pg['n'] = pg['CANCER_GROUP'].apply(len)
fig = make_subplots(rows=1,cols=2,subplot_titles=('Groups per Patient','Multi-Group Patients'),
    specs=[[{'type':'bar'},{'type':'table'}]])
gc = pg['n'].value_counts().sort_index()
fig.add_trace(go.Bar(x=gc.index.astype(str),y=gc.values,marker_color='steelblue',text=gc.values,textposition='outside',showlegend=False),row=1,col=1)
mp = pg[pg['n']>1].copy()
mp['Groups'] = mp['CANCER_GROUP'].apply(lambda x:', '.join(sorted(str(v) for v in x)))
mp = mp.sort_values('n',ascending=False).head(20)
if len(mp)>0:
    t = mp[['PATIENT_ID','n','Groups']].head(15)
    fig.add_trace(go.Table(header=dict(values=['Patient','N','Groups'],fill_color='lightblue',align='left',font=dict(size=10)),
        cells=dict(values=[t['PATIENT_ID'],t['n'],t['Groups']],align='left',height=22,font=dict(size=9))),row=1,col=2)
fig.update_layout(height=500,title='Multi-Cancer-Group Analysis')
fig.show()


In [51]:
# Step 9a: Multi-Cancer-Group Analysis -- How many patients have samples in different cancer groups.
m2 = pg[pg['n']==2].copy()
if len(m2)>0:
    m2['gl'] = m2['CANCER_GROUP'].apply(lambda x:sorted(str(v) for v in x))
    m2['g1'] = m2['gl'].apply(lambda x:x[0])
    m2['g2'] = m2['gl'].apply(lambda x:x[1])
    co = pd.crosstab(m2['g1'],m2['g2'])
    fig = go.Figure(data=go.Heatmap(z=co.values,x=co.columns,y=co.index,text=co.values,
        texttemplate='%{text}',colorscale='Blues'))
    fig.update_layout(title='Cancer Group Co-occurrence (2 groups)',xaxis_tickangle=-45,height=450)
    fig.show()
else: print('No patients with exactly 2 groups.')


In [52]:
# Step 9a: Multi-Cancer-Group Analysis -- How many patients have samples in different cancer groups.
print(f'Multi-Group patients: {(pg["n"]>1).sum()} / {pg["PATIENT_ID"].nunique()}')
print(f'Max groups: {pg["n"].max()}')
if len(m2)>0: print(f'Exactly 2 groups: {len(m2)}')


Multi-Group patients: 111 / 2870
Max groups: 4
Exactly 2 groups: 103
